# Chapter 4: The Dirac Delta Function

Companion notebook to Section 4 of the study guide (Kundu, Cohen & Dowling 2016, Appendix B.4).
Write your own notes under each heading as you study.

Before committing: **Restart** the kernel, then **Run All**, then **save**, so the notebook runs from top to bottom.

In [ ]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import sympy as sp
from scipy.integrate import trapezoid
from scipy.special import erf

from ch04_tools import (coarse_grain, diffuse_by_convolution, gaussian_delta, heat_kernel,
                        nash_cascade_iuh, profile_moments, rainfall_runoff, sifting_integral,
                        tracer_concentration, velocity_and_dispersion)

FIG_DIR = Path("figures")   # figures worth showing on GitHub
FIG_DIR.mkdir(exist_ok=True)

BLUE, ORANGE = "#2a78d6", "#eb6834"        # categorical slots 1 and 2
RAMP = ["#86b6ef", "#2a78d6", "#104281"]   # ordered series: light -> dark
GRAY = "#52514e"                           # reference curves

## 4.1 Definition and intuition

$\delta$ is defined by what it does inside an integral: $\displaystyle\int f(x)\,\delta(x-x_0)\,dx = f(x_0)$ (B.4.4).
Picture it as the limit of unit-area Gaussians, $\delta_\sigma(x) = \dfrac{1}{\sqrt{2\pi}\,\sigma}e^{-x^2/2\sigma^2}$ as $\sigma\to0$ (B.4.1).

Properties: $x\,\delta(x-a) = a\,\delta(x-a)$; $\delta(\alpha x) = \delta(x)/|\alpha|$; $\delta(g(x)) = \sum_n \delta(x-x_n)/|g'(x_n)|$;
$\int f\,\delta'(x-x_0)\,dx = -f'(x_0)$; $H' = \delta$; units $[\delta(x)] = 1/[x]$.

*My notes:*

In [ ]:
x = np.linspace(-20, 20, 400001)
f = lambda s: np.cos(s) + s**2          # any smooth test function
x0 = 1.3

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
for sigma, color, style in zip([1.0, 0.5, 0.25], RAMP, ["-", "--", "-."]):
    axes[0].plot(x, gaussian_delta(x, sigma), style, color=color, label=f"sigma = {sigma}")
    print(f"sigma = {sigma:4.2f}: area = {trapezoid(gaussian_delta(x, sigma), x):.6f}")
axes[0].set(xlim=(-3, 3), xlabel="x", ylabel="delta_sigma(x)  (1/[x])",
            title="(a) narrower and taller, always unit area")
axes[0].legend(fontsize=8)

sigmas = np.logspace(0, -2, 30)
values = [sifting_integral(f, x, x0, s) for s in sigmas]
axes[1].semilogx(sigmas, values, "o-", color=BLUE, ms=4)
axes[1].axhline(f(x0), color=GRAY, ls=":", lw=1.5)
axes[1].text(0.012, f(x0) + 0.03, f"f(x0) = {f(x0):.4f}", ha="right", va="bottom", fontsize=8, color=GRAY)
axes[1].invert_xaxis()
axes[1].set(xlabel="sigma (decreasing)", ylabel="integral of f delta_sigma",
            title="(b) sifting: the integral picks out f(x0)")
fig.tight_layout()
fig.savefig(FIG_DIR / "ch04_delta_family.png", dpi=200)
plt.show()

In [ ]:
# The properties with SymPy's DiracDelta
x = sp.symbols("x", real=True)
print("sifting     :", sp.integrate(sp.sin(x) * sp.DiracDelta(x - sp.pi / 2), (x, -sp.oo, sp.oo)))   # 1
print("scaling     :", sp.integrate(sp.exp(x) * sp.DiracDelta(3 * x - 6), (x, -sp.oo, sp.oo)))      # exp(2)/3
print("composition :", sp.DiracDelta(x**2 - 9).expand(diracdelta=True, wrt=x))               # roots +-3
print("derivative  :", sp.integrate(x**3 * sp.DiracDelta(x - 2, 1), (x, -sp.oo, sp.oo)))           # -12
print("outside     :", sp.integrate(sp.cos(x) * sp.DiracDelta(x - 7), (x, 0, 5)))            # 0

### Example B.4: a moving source

$\delta(x - ct)$ is a point source moving at speed $c$. Integrating over $x$ simply replaces $x$ by $ct$ in the rest of the integrand.

*My notes:*

In [ ]:
x = sp.symbols("x", real=True)
t, c, x0, r0, k = sp.symbols("t c x_0 r_0 k", positive=True)
F = sp.Function("F")
integrand = F(x) * ((x0 - x)**2 + r0**2)**sp.Rational(-1, 2) * sp.exp(sp.I * k * x)
print(sp.integrate(integrand * sp.DiracDelta(x - c * t), (x, -sp.oo, sp.oo)))
# F(c t) [(x0 - c t)^2 + r0^2]^(-1/2) exp(i k c t)

## 4.2 Green's functions: responses to impulses

For a linear problem $\mathcal L f = s$, write the source as a sum of impulses; then $f = \iint G\,s\,d\xi\,d\tau$, where $\mathcal L G = \delta(x-\xi)\,\delta(t-\tau)$.

Diffusion on an infinite line: $G(x,t) = \dfrac{1}{\sqrt{4\pi\kappa t}}\,e^{-x^2/4\kappa t}$, a Gaussian with variance $2\kappa t$, and $c(x,t) = \int G(x-\xi,t)\,c_0(\xi)\,d\xi$.

*My notes:*

In [ ]:
x = sp.symbols("x", real=True)
t, kappa = sp.symbols("t kappa", positive=True)
G = sp.exp(-x**2 / (4 * kappa * t)) / sp.sqrt(4 * sp.pi * kappa * t)
print("G_t - kappa G_xx =", sp.simplify(sp.diff(G, t) - kappa * sp.diff(G, x, 2)))   # 0
print("area             =", sp.integrate(G, (x, -sp.oo, sp.oo)))                             # 1
print("variance         =", sp.simplify(sp.integrate(x**2 * G, (x, -sp.oo, sp.oo))))         # 2 kappa t

In [ ]:
x = np.linspace(-80, 80, 1601)            # [m]

fig, axes = plt.subplots(1, 2, figsize=(11, 3.6))
a, kt = 2.0, 10.0                           # step of half-height 2 m; kappa t = 10 m^2
c = diffuse_by_convolution(a * np.sign(x), x, kt)
axes[0].plot(x, a * erf(x / np.sqrt(4 * kt)), color=GRAY, lw=1, label="a erf(x / sqrt(4 kappa t))")
axes[0].plot(x[::40], c[::40], "o", color=BLUE, ms=5, label="step convolved with G")
axes[0].set(xlim=(-40, 40), xlabel="x (m)", ylabel="z (m)",
            title="(a) step convolved with G = the erf of Chapter 2")
axes[0].legend(loc="upper left", fontsize=8)

pulse = np.where(np.abs(x) <= 10, 1.0, 0.0)   # a 20 m wide, 1 m thick sediment pulse
axes[1].plot(x, pulse, color=GRAY, lw=1, label="initial pulse")
print(f"initial pulse:  volume = {trapezoid(pulse, x):.2f} m^2, peak = 1.00 m")
for kt, color, style in zip([10, 50, 200], RAMP, ["-", "--", "-."]):
    c = diffuse_by_convolution(pulse, x, kt)
    axes[1].plot(x, c, style, color=color, label=f"kappa t = {kt} m^2")
    print(f"kappa t = {kt:3d} m^2: volume = {trapezoid(c, x):.2f} m^2, peak = {c.max():.2f} m")
axes[1].set(xlim=(-60, 60), xlabel="x (m)", ylabel="thickness (m)",
            title="(b) a pulse spreads and flattens, volume conserved")
axes[1].legend(loc="upper right", fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "ch04_green_function.png", dpi=200)
plt.show()

### Three routes to the same solution

The erf solution of Chapter 2 comes from (i) a similarity variable, (ii) convolving the step with $G$ (panel a above), or (iii) a Fourier transform.

*My notes:*

## 4.3 Research connection: impulses and responses in Earth-surface systems

- **Tracer injections:** $c = \dfrac{M/A}{\sqrt{4\pi K_x t}}\exp\!\left[-\dfrac{(x-Ut)^2}{4K_xt}\right]$; the centroid moves at $U$ and $K_x = \tfrac12\,d\sigma^2/dt$.
- **Unit hydrographs:** $Q(t) = \int_0^t u(t-\tau)\,p(\tau)\,d\tau$, with $u$ the response to $p = \delta(t)$.
- **Sediment pulses** and **signal shredding:** a storm train $\sum a_i\delta(t-t_i)$ leaves a linear landscape as $\sum a_i G(t-t_i)$.
- **From grains to fields:** $n_\sigma(x,t) = \sum_p \delta_\sigma(x - x_p(t))$.

*My notes:*

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))

# (a) Tracer test: 10 kg of salt across a 15 m^2 section, U = 0.8 m/s, Kx = 2 m^2/s
x = np.linspace(0, 4000, 40001)
times = [1200.0, 3600.0]                                  # 20 and 60 minutes [s]
moments = []
for t, color, style in zip(times, [RAMP[0], RAMP[2]], ["-", "--"]):
    c = tracer_concentration(x, t, M=10.0, A=15.0, U=0.8, Kx=2.0)
    moments.append(profile_moments(x, c))
    axes[0].plot(x / 1e3, c * 1e3, style, color=color, label=f"t = {t / 60:.0f} min")
(_, m1, v1), (_, m2, v2) = moments
U, Kx = velocity_and_dispersion(times[0], m1, v1, times[1], m2, v2)
print(f"tracer: U = {U:.2f} m/s, Kx = {Kx:.2f} m^2/s")   # 0.80, 2.00
axes[0].set(xlabel="distance downstream (km)", ylabel="concentration (mg/L)",
            title="(a) tracer slug: drifts at U, variance grows as 2 Kx t")
axes[0].legend(fontsize=8)

# (b) Unit hydrograph: rainfall convolved with two impulse responses of equal mean lag (3 h)
dt = 0.05
t = np.arange(0, 24, dt)                                   # [h]
p = np.where((t >= 1) & (t < 3), 10.0, 0.0) + np.where((t >= 6) & (t < 7), 5.0, 0.0)   # [mm/h]
axes[1].fill_between(t, p, step="post", color="#cde2fb", label="rainfall")
for (n, k, name), color, style in zip([(1, 3.0, "linear reservoir, k = 3 h"),
                                       (3, 1.0, "Nash cascade, n = 3, k = 1 h")],
                                      [BLUE, ORANGE], ["-", "--"]):
    Q = rainfall_runoff(p, dt, nash_cascade_iuh(t, k, n))
    axes[1].plot(t, Q, style, color=color, label=name)
    print(f"{name}: peak {Q.max():.2f} mm/h at {t[np.argmax(Q)]:.1f} h, "
          f"runoff volume {trapezoid(Q, t):.1f} mm of {trapezoid(p, t):.1f} mm rain")
axes[1].set(xlabel="time (h)", ylabel="rate (mm/h)", title="(b) rainfall in, smoothed and lagged discharge out")
axes[1].legend(fontsize=8)

# (c) From grains to a continuum density
rng = np.random.default_rng(42)
grains = np.concatenate([rng.uniform(0, 100, 250), rng.normal(60, 4, 150)])   # positions [m]
x = np.linspace(-20, 120, 2801)
for sigma, color, style in zip([0.5, 5.0], [RAMP[0], RAMP[2]], ["-", "--"]):
    n = coarse_grain(x, grains, sigma)
    axes[2].plot(x, n, style, color=color, lw=1.2, label=f"sigma = {sigma} m")
    print(f"sigma = {sigma} m: integral of n = {trapezoid(n, x):.1f} grains")   # 400
axes[2].plot(grains, np.full(grains.size, -0.4), "|", color=GRAY, ms=8)
axes[2].set(xlim=(0, 100), xlabel="x (m)", ylabel="grains per metre",
            title="(c) grains to a field: sigma sets the detail")
axes[2].legend(fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "ch04_impulse_responses.png", dpi=200)
plt.show()

## Exercises 4.1-4.7

*Work space: solve each exercise here, then check against Appendix C of the study guide.*